# Notebook 1: exploración, refinado y modelo de datos

**Obligatorio Herramientas de Software para Big Data, Parte 1. Vuelos regulares de Brasil, 2025.**
Grupo: Pilar Fernández, Jerónimo Brum y Fernando Tonarelli.

Este notebook va desde los archivos crudos que dejó NiFi en la zona *landing* hasta las tablas del modelo creadas y pobladas en Hive. El notebook 2 (analista) arranca de esas tablas y no lee nada de HDFS directamente.

| Paso | Qué hace | Salida |
|---|---|---|
| 1 | Lectura de las 6 fuentes desde `/lnd` | DataFrames crudos |
| 2 | Exploración: filas, columnas, esquema, nulos, duplicados, claves | Perfil de cada tabla |
| 3 | Validaciones a mano | Conteos contra los valores conocidos |
| 4 | Reglas de negocio y refinado | Tablas limpias |
| 5 | Guardado en `/rfn` y control de filas | Parquet en la zona refinada |
| 6 | Modelo estrella | Hechos y dimensiones |
| 7 | DDL de Hive y carga | Base `vuelos_brasil` |
| 8 | Validación final en Hive | Conteos, particiones, tipo de tabla |

Todo se hace con Spark. Pandas no se usa en este notebook.

## 0. Configuración

Antes de correr, en la terminal de la VM:

```
docker exec namenode hdfs dfs -chmod -R 777 /rfn
```

Hive y Spark escriben con usuarios distintos; sin ese permiso, la escritura en `/rfn` falla.

**Lo único que hay que cambiar al pasar el notebook a otra VM es `INGESTA`**: la fecha de la carpeta `ingesta=AAAAMMDD` que creó NiFi (se ve con `hdfs dfs -ls /lnd/vuelos_brasil/anac_vra`).

In [2]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("nb1_vuelos_brasil_refinado")
         .enableHiveSupport()
         .getOrCreate())

INGESTA = "20261112"      # carpeta ingesta=AAAAMMDD que dejó NiFi en landing
VERSION = "v1"            # versión del refinado; cambia si cambian las reglas
PREFIX = ""               # vacío = HDFS por defecto de la VM
ENCODING_AERODROMOS = "ISO-8859-1"   # el cadastro de aeródromos viene en Latin-1

LND = f"{PREFIX}/lnd/vuelos_brasil"
RFN = f"{PREFIX}/rfn/vuelos_brasil/{VERSION}"
MODELO = f"{RFN}/modelo"
DB = "vuelos_brasil"

def ruta_lnd(fuente, archivo="*"):
    return f"{LND}/{fuente}/ingesta={INGESTA}/{archivo}"

print("Landing :", LND)
print("Refined :", RFN)
print("Modelo  :", MODELO)

Landing : /lnd/vuelos_brasil
Refined : /rfn/vuelos_brasil/v1
Modelo  : /rfn/vuelos_brasil/v1/modelo


## 1. Lectura de las fuentes

| Fuente | Carpeta en landing | Formato |
|---|---|---|
| Vuelos (ANAC VRA), un CSV por mes | `anac_vra` | `;`, UTF-8 con BOM, línea "Atualizado em" antes del encabezado |
| Empresas aéreas brasileñas (ANAC) | `anac_empresas_br` | igual que los vuelos |
| Empresas aéreas extranjeras (ANAC) | `anac_empresas_ext` | igual que los vuelos |
| Aeródromos públicos (ANAC) | `anac_aerodromos` | `;` sin comillas, **Latin-1**, línea "Atualizado em" |
| Aeropuertos del mundo (OurAirports) | `ourairports_airports` | `,` con comillas, UTF-8, encabezado en la línea 1 |
| Países (OurAirports) | `ourairports_countries` | igual que aeropuertos |

Los archivos de ANAC tienen una primera línea `Atualizado em: <fecha>` y recién la segunda es el encabezado. Spark no tiene una opción para saltear líneas, así que `leer_anac` lee todo sin encabezado, toma el encabezado de la primera fila que no es "Atualizado em" y descarta esa fila y la del encabezado de cada archivo. De paso guarda la fecha de actualización de cada archivo.

Las columnas se renombran a `snake_case` (Hive pasa todo a minúscula y no acepta espacios ni tildes).

In [2]:
def leer_anac(ruta, columnas, encoding="UTF-8", quote='"'):
    # Lee un CSV de ANAC: descarta la línea "Atualizado em" y el encabezado de cada archivo.
    # La primera línea ("Atualizado em") tiene una sola columna; si se dejara inferir el esquema,
    # Spark leería todo el archivo con una columna. Por eso se pasa el esquema explícito.
    esquema = T.StructType([T.StructField(f"_c{i}", T.StringType()) for i in range(len(columnas))])
    crudo = (spark.read
             .schema(esquema)
             .option("sep", ";")
             .option("header", False)
             .option("quote", quote)
             .option("encoding", encoding)
             .csv(ruta)
             .withColumn("archivo_origen", F.regexp_extract(F.input_file_name(), r"([^/]+)$", 1)))
    # Spark lee los campos vacíos como nulos: las comparaciones tienen que ser null-safe,
    # si no se pierden las filas con la primera columna vacía.
    es_actualizado = F.coalesce(F.col("_c0").contains("Atualizado em"), F.lit(False))
    encabezado = crudo.filter(~es_actualizado).first()
    n_cols = len(columnas)
    assert encabezado[n_cols - 1] is not None, "El encabezado tiene menos columnas que las esperadas"
    print("Encabezado original:", [encabezado[i] for i in range(n_cols)])
    print("Actualización:", [r[0].replace("﻿", "").strip()
                             for r in crudo.filter(es_actualizado).select("_c0").distinct().collect()])
    datos = crudo.filter(~es_actualizado & ~F.col("_c0").eqNullSafe(encabezado[0]))
    for i, nombre in enumerate(columnas):
        datos = datos.withColumnRenamed(f"_c{i}", nombre)
    return datos

COLS_VRA = ["icao_empresa", "numero_voo", "codigo_di", "tipo_linha", "icao_origem", "icao_destino",
            "partida_prevista", "partida_real", "chegada_prevista", "chegada_real",
            "situacao_voo", "codigo_justificativa"]
COLS_EMPRESAS = ["icao", "iata", "razao_social", "servico", "endereco", "cidade", "uf", "cep",
                 "telefone", "email", "instrumento", "data_instrumento", "ativa"]
COLS_AERODROMOS = ["codigo_oaci", "ciad", "nome", "municipio", "uf", "municipio_servido", "uf_servido",
                   "lat_geopoint", "lon_geopoint", "latitude_gms", "longitude_gms", "altitude",
                   "operacao_diurna", "operacao_noturna", "situacao", "validade_registro",
                   "portaria_registro", "link_portaria"]

In [3]:
vuelos_raw = leer_anac(ruta_lnd("anac_vra", "*.csv"), COLS_VRA).cache()   # se reusa en toda la exploración
emp_br_raw = leer_anac(ruta_lnd("anac_empresas_br", "*.csv"), COLS_EMPRESAS)
emp_ext_raw = leer_anac(ruta_lnd("anac_empresas_ext", "*.csv"), COLS_EMPRESAS)
aerodromos_raw = leer_anac(ruta_lnd("anac_aerodromos", "*.csv"), COLS_AERODROMOS,
                           encoding=ENCODING_AERODROMOS, quote="")

def leer_ourairports(ruta):
    return (spark.read
            .option("header", True)
            .option("quote", '"')
            .option("escape", '"')
            .option("multiLine", True)
            .csv(ruta))

airports_raw = leer_ourairports(ruta_lnd("ourairports_airports", "airports.csv"))
countries_raw = leer_ourairports(ruta_lnd("ourairports_countries", "countries.csv"))

Encabezado original: ['ICAO Empresa Aérea', 'Número Voo', 'Código Autorização (DI)', 'Código Tipo Linha', 'ICAO Aeródromo Origem', 'ICAO Aeródromo Destino', 'Partida Prevista', 'Partida Real', 'Chegada Prevista', 'Chegada Real', 'Situação Voo', 'Código Justificativa']
Actualización: ['Atualizado em: 2026-10-28']
Encabezado original: ['ICAO', 'Estrangeira', 'Razao', 'Servico', 'Endereco', 'Cidade', 'UF', 'CEP', 'Telefone', 'Email', 'Instrumento', 'Data', 'Ativa']
Actualización: ['Atualizado em: 2026-04-06']
Encabezado original: ['ICAO', 'Estrangeira', 'Razao', 'Servico', 'Endereco', 'Cidade', 'UF', 'CEP', 'Telefone', 'Email', 'Instrumento', 'Data', 'Ativa']
Actualización: ['Atualizado em: 2026-11-11']
Encabezado original: ['Código OACI', 'CIAD', 'Nome', 'Município', 'UF', 'Município Servido', 'UF Servido', 'LATGEOPOINT', 'LONGEOPOINT', 'Latitude', 'Longitude', 'Altitude', 'Operação Diurna', 'Operação Noturna', 'Situação', 'Validade do Registro', 'Portaria de Registro', 'Link Portaria']


## 2. Exploración de cada tabla

Para cada tabla: primeras filas, cantidad de filas y columnas, esquema, nulos o vacíos por columna, filas duplicadas y unicidad de la clave. En esta etapa todas las columnas son texto: los tipos se asignan en el refinado.

`valores_basura` cuenta valores que en estas fuentes significan "sin dato" aunque no estén vacíos ("N/I", "..", "000", "null", etc.).

In [4]:
BASURA = ["N/I", "N/A", "..", ".", "-", "null", "0", "00", "000", "0000000", "00.000-000", "***", "**"]

def perfilar(df, nombre, clave=None, basura=True):
    n = df.count()
    print(f"==================== {nombre} ====================")
    print(f"Filas: {n:,} | Columnas: {len(df.columns)}")
    df.printSchema()
    df.show(5, truncate=40)
    nulos = df.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1)
                             .otherwise(0)).alias(c) for c in df.columns]).first().asDict()
    filas = []
    for c in df.columns:
        b = (df.filter(F.trim(F.col(c).cast("string")).isin(BASURA)).count() if basura else 0)
        filas.append((c, int(nulos[c] or 0), round(100 * (nulos[c] or 0) / max(n, 1), 2), b))
    spark.createDataFrame(filas, "columna string, nulos_o_vacios long, pct double, valores_basura long") \
         .show(len(filas), truncate=False)
    print("Filas duplicadas completas:", n - df.dropDuplicates().count())
    if clave:
        rep = df.filter(F.concat_ws("", *clave) != "").groupBy(*clave).count().filter("count > 1")
        print(f"Valores de la clave {clave} repetidos:", rep.count())
        rep.orderBy(F.desc("count")).show(5, truncate=False)

### 2.1 Vuelos (VRA)

| Columna | Significado |
|---|---|
| icao_empresa | Código ICAO de la aerolínea (3 letras) |
| numero_voo | Número de vuelo, con ceros adelante ("0057") |
| codigo_di | Código de autorización (DI); 0 en la gran mayoría |
| tipo_linha | N nacional, I internacional, C carga doméstica, G carga internacional |
| icao_origem, icao_destino | Código ICAO de los aeropuertos de salida y llegada |
| partida_prevista, partida_real | Salida programada y real, en hora de Brasília |
| chegada_prevista, chegada_real | Llegada programada y real, en hora de Brasília |
| situacao_voo | REALIZADO o CANCELADO |
| codigo_justificativa | Motivo de atraso o cancelación; "N/A" si no hay |

Clave candidata del vuelo: empresa + número de vuelo + fecha y hora de referencia + origen + destino. La fecha de referencia es la salida prevista y, si no existe, la salida real (ver la regla 2 de la sección 4).

In [5]:
perfilar(vuelos_raw, "VUELOS (VRA)", basura=False)
vuelos_raw.groupBy("archivo_origen").count().orderBy("archivo_origen").show(12)
vuelos_raw.groupBy("situacao_voo").count().orderBy(F.desc("count")).show()
vuelos_raw.groupBy("tipo_linha").count().orderBy(F.desc("count")).show()
vuelos_raw.groupBy("codigo_justificativa").count().orderBy(F.desc("count")).show(10)
vuelos_raw.groupBy("codigo_di").count().orderBy(F.desc("count")).show(15)
# Formato de las fechas: algunas traen fracción de segundo
cols_fecha = ["partida_prevista", "partida_real", "chegada_prevista", "chegada_real"]
for c in cols_fecha:
    print(f"{c}: con fracción de segundo = {vuelos_raw.filter(F.col(c).contains('.')).count():,}"
          f" | con el texto 'null' = {vuelos_raw.filter(F.col(c) == 'null').count():,}")
print("Vuelos sin salida prevista ('null'), por situación, tipo y código DI:")
sin_prevista = vuelos_raw.filter(F.col("partida_prevista") == "null")
sin_prevista.groupBy("situacao_voo").count().show()
sin_prevista.groupBy("tipo_linha").count().orderBy(F.desc("count")).show()
sin_prevista.groupBy("codigo_di").count().orderBy(F.desc("count")).show()
print("Cancelados con salida real informada:",
      vuelos_raw.filter((F.col("situacao_voo") == "CANCELADO") & (F.col("partida_real") != "null")).count())

==================== VUELOS (VRA) ====================
Filas: 1,005,807 | Columnas: 13
root
 |-- icao_empresa: string (nullable = true)
 |-- numero_voo: string (nullable = true)
 |-- codigo_di: string (nullable = true)
 |-- tipo_linha: string (nullable = true)
 |-- icao_origem: string (nullable = true)
 |-- icao_destino: string (nullable = true)
 |-- partida_prevista: string (nullable = true)
 |-- partida_real: string (nullable = true)
 |-- chegada_prevista: string (nullable = true)
 |-- chegada_real: string (nullable = true)
 |-- situacao_voo: string (nullable = true)
 |-- codigo_justificativa: string (nullable = true)
 |-- archivo_origen: string (nullable = false)

+------------+----------+---------+----------+-----------+------------+-------------------+-------------------+-------------------+-------------------+------------+--------------------+--------------+
|icao_empresa|numero_voo|codigo_di|tipo_linha|icao_origem|icao_destino|   partida_prevista|       partida_real|   chegada_p

### 2.2 Empresas aéreas brasileñas y extranjeras

Los dos archivos tienen las mismas 13 columnas.

| Columna | Significado |
|---|---|
| icao | Código ICAO de 3 letras; solo lo tienen las empresas de transporte aéreo |
| iata | Pese a que el archivo la llama "Estrangeira", es el código IATA (G3 Gol, AD Azul) |
| razao_social | Razón social |
| servico | Servicios autorizados (en extranjeras: REGULAR, NÃO REGULAR u OFFLINE) |
| endereco, cidade, uf, cep | Dirección de la sede o del representante en Brasil |
| telefone, email | Contacto (dato personal en algunos casos: no pasa a refined) |
| instrumento, data_instrumento | Portaria que habilita a la empresa y su fecha (M/d/yyyy h:mm:ss AM) |
| ativa | Estado; siempre ATIVA |

In [6]:
perfilar(emp_br_raw, "EMPRESAS BRASILEÑAS", clave=["icao"])
perfilar(emp_ext_raw, "EMPRESAS EXTRANJERAS", clave=["icao"])
print("ICAO en los dos archivos a la vez:",
      emp_br_raw.select("icao").filter("icao != ''").intersect(emp_ext_raw.select("icao")).count())

==================== EMPRESAS BRASILEÑAS ====================
Filas: 729 | Columnas: 14
root
 |-- icao: string (nullable = true)
 |-- iata: string (nullable = true)
 |-- razao_social: string (nullable = true)
 |-- servico: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- cep: string (nullable = true)
 |-- telefone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- instrumento: string (nullable = true)
 |-- data_instrumento: string (nullable = true)
 |-- ativa: string (nullable = true)
 |-- archivo_origen: string (nullable = false)

+----+----+----------------------------------------+---------------------------------+----------------------------------------+---------------------+---+----------+---------------+------------------------------+----------------------------------------+---------------------+-----+---------------------------------+
|icao|iata|                          

### 2.3 Aeródromos públicos (ANAC)

| Columna | Significado |
|---|---|
| codigo_oaci | Código ICAO de 4 letras (OACI en portugués) |
| ciad | Identificador del aeródromo en Brasil: sigla del estado + 4 dígitos |
| nome, municipio, uf | Nombre, municipio y estado (nombre completo, no sigla) |
| municipio_servido, uf_servido | Ciudad y estado a los que da servicio |
| lat_geopoint, lon_geopoint | Coordenadas en grados decimales |
| latitude_gms, longitude_gms | Las mismas coordenadas en grados, minutos y segundos |
| altitude | Altura en metros, con coma decimal |
| operacao_diurna, operacao_noturna | Tipo de operación permitida |
| situacao | Cadastrado o Interditado |
| validade_registro, portaria_registro, link_portaria | Datos del registro |

In [7]:
perfilar(aerodromos_raw, "AERÓDROMOS PÚBLICOS", clave=["codigo_oaci"], basura=False)
aerodromos_raw.groupBy("uf").count().orderBy(F.desc("count")).show(30, truncate=False)
aerodromos_raw.groupBy("situacao").count().show()

==================== AERÓDROMOS PÚBLICOS ====================
Filas: 496 | Columnas: 19
root
 |-- codigo_oaci: string (nullable = true)
 |-- ciad: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio_servido: string (nullable = true)
 |-- uf_servido: string (nullable = true)
 |-- lat_geopoint: string (nullable = true)
 |-- lon_geopoint: string (nullable = true)
 |-- latitude_gms: string (nullable = true)
 |-- longitude_gms: string (nullable = true)
 |-- altitude: string (nullable = true)
 |-- operacao_diurna: string (nullable = true)
 |-- operacao_noturna: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- validade_registro: string (nullable = true)
 |-- portaria_registro: string (nullable = true)
 |-- link_portaria: string (nullable = true)
 |-- archivo_origen: string (nullable = false)

+-----------+------+------------------------+--------------------+--------+--------

### 2.4 Aeropuertos y países (OurAirports)

Diccionario oficial: https://ourairports.com/help/data-dictionary.html. Las columnas que se usan son `ident`, `gps_code`, `icao_code` (códigos), `type` (tamaño), `name`, `iso_country`, `iso_region` (estado, por ejemplo BR-SP), `municipality` y las coordenadas. En `countries`: `code` (ISO de 2 letras), `name` y `continent`.

Ojo: en `continent`, "NA" es América del Norte, y en `countries.code`, "NA" es Namibia. Spark lo lee como texto; pandas lo convertiría en nulo.

In [8]:
perfilar(airports_raw, "AEROPUERTOS (OurAirports)", clave=["ident"], basura=False)
airports_raw.filter("iso_country = 'BR'").groupBy("type").count().orderBy(F.desc("count")).show()
for c in ["ident", "gps_code", "icao_code"]:
    rep = airports_raw.filter(F.col(c).isNotNull()).groupBy(c).count().filter("count > 1").count()
    print(f"{c}: valores repetidos = {rep}")
perfilar(countries_raw, "PAÍSES (OurAirports)", clave=["code"], basura=False)

==================== AEROPUERTOS (OurAirports) ====================
Filas: 86,351 | Columnas: 19
root
 |-- id: string (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: string (nullable = true)
 |-- longitude_deg: string (nullable = true)
 |-- elevation_ft: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

+------+-----+-------------+--------------------+-----------------+------------------+------------+---------+--

## 3. Validaciones a mano

Se comparan los conteos con valores revisados a mano en los archivos originales (abiertos en el navegador o en un editor) el 10 y 11/11. Las filas de los 12 archivos de vuelos se contaron el 16/11 fuera de Spark, con un script aparte que cuenta las líneas de cada CSV menos la línea "Atualizado em" y el encabezado. Si Spark llega al mismo número, la lectura no perdió ni duplicó filas. Si un valor no coincide, se marca **DIFERENTE** y hay que revisar antes de seguir.

Además se prueba cómo cruzan los vuelos con las otras tablas: esto decide qué columna de OurAirports usar como clave.

In [9]:
# Filas de cada CSV de vuelos contadas fuera de Spark: líneas del archivo menos "Atualizado em" y el encabezado
ESPERADOS_VRA = {"VRA_20251.csv": 89616, "VRA_20252.csv": 78930, "VRA_20253.csv": 82048,
                 "VRA_20254.csv": 80692, "VRA_20255.csv": 82565, "VRA_20256.csv": 80974,
                 "VRA_20257.csv": 87725, "VRA_20258.csv": 84583, "VRA_20259.csv": 82156,
                 "VRA_202510.csv": 85709, "VRA_202511.csv": 82245, "VRA_202512.csv": 88564}
ESPERADO_SITUACAO_ENERO = {"REALIZADO": 86376, "CANCELADO": 3240}
ESPERADO_TIPO_ENERO = {"N": 70984, "I": 15266, "G": 1808, "C": 1558}

def chequear(nombre, obtenido, esperado):
    estado = "OK" if obtenido == esperado else "DIFERENTE"
    print(f"[{estado}] {nombre}: obtenido {obtenido:,} / esperado {esperado:,}")

por_archivo = {r["archivo_origen"]: r["count"] for r in vuelos_raw.groupBy("archivo_origen").count().collect()}
for archivo, esperado in ESPERADOS_VRA.items():
    chequear(f"vuelos en {archivo}", por_archivo.get(archivo, 0), esperado)   # si falta un archivo, da 0
chequear("vuelos en el año", sum(por_archivo.values()), sum(ESPERADOS_VRA.values()))

enero = vuelos_raw.filter(F.col("archivo_origen") == "VRA_20251.csv")
for k, v in ESPERADO_SITUACAO_ENERO.items():
    chequear(f"enero, situacao {k}", enero.filter(F.col("situacao_voo") == k).count(), v)
for k, v in ESPERADO_TIPO_ENERO.items():
    chequear(f"enero, tipo_linha {k}", enero.filter(F.col("tipo_linha") == k).count(), v)

chequear("empresas brasileñas", emp_br_raw.count(), 729)
chequear("empresas extranjeras", emp_ext_raw.count(), 149)
chequear("aeródromos públicos", aerodromos_raw.count(), 496)
print("Aeropuertos OurAirports:", airports_raw.count(), "(86.351 el 11/11; cambia todos los días)")
print("Países OurAirports:", countries_raw.count(), "(249 el 11/11)")

[OK] vuelos en VRA_20251.csv: obtenido 89,616 / esperado 89,616
[OK] vuelos en VRA_20252.csv: obtenido 78,930 / esperado 78,930
[OK] vuelos en VRA_20253.csv: obtenido 82,048 / esperado 82,048
[OK] vuelos en VRA_20254.csv: obtenido 80,692 / esperado 80,692
[OK] vuelos en VRA_20255.csv: obtenido 82,565 / esperado 82,565
[OK] vuelos en VRA_20256.csv: obtenido 80,974 / esperado 80,974
[OK] vuelos en VRA_20257.csv: obtenido 87,725 / esperado 87,725
[OK] vuelos en VRA_20258.csv: obtenido 84,583 / esperado 84,583
[OK] vuelos en VRA_20259.csv: obtenido 82,156 / esperado 82,156
[OK] vuelos en VRA_202510.csv: obtenido 85,709 / esperado 85,709
[OK] vuelos en VRA_202511.csv: obtenido 82,245 / esperado 82,245
[OK] vuelos en VRA_202512.csv: obtenido 88,564 / esperado 88,564
[OK] vuelos en el año: obtenido 1,005,807 / esperado 1,005,807
[OK] enero, situacao REALIZADO: obtenido 86,376 / esperado 86,376
[OK] enero, situacao CANCELADO: obtenido 3,240 / esperado 3,240
[OK] enero, tipo_linha N: obtenido 7

Un vuelo concreto revisado a mano en el CSV original: el primer registro de enero es Air Europa (AEA) 0057, de Madrid (LEMD) a Guarulhos (SBGR), previsto 2025-01-29 19:55, salió 20:45. El atraso calculado a mano es 50 minutos. La misma cuenta se repite con Spark en el refinado (sección 4).

In [10]:
vuelos_raw.filter((F.col("icao_empresa") == "AEA") & (F.col("numero_voo") == "0057")
                  & (F.col("partida_prevista") == "2025-01-29 19:55:00")).show(truncate=False)

+------------+----------+---------+----------+-----------+------------+-------------------+-------------------+-------------------+-------------------+------------+--------------------+--------------+
|icao_empresa|numero_voo|codigo_di|tipo_linha|icao_origem|icao_destino|partida_prevista   |partida_real       |chegada_prevista   |chegada_real       |situacao_voo|codigo_justificativa|archivo_origen|
+------------+----------+---------+----------+-----------+------------+-------------------+-------------------+-------------------+-------------------+------------+--------------------+--------------+
|AEA         |0057      |0        |I         |LEMD       |SBGR        |2025-01-29 19:55:00|2025-01-29 20:45:00|2025-01-30 06:30:00|2025-01-30 06:47:00|REALIZADO   |N/A                 |VRA_20251.csv |
+------------+----------+---------+----------+-----------+------------+-------------------+-------------------+-------------------+-------------------+------------+--------------------+-----------

In [11]:
# Cruce de los vuelos con las otras tablas
empresas_icao = emp_br_raw.select("icao").union(emp_ext_raw.select("icao")).filter("icao != ''").distinct()
emp_vuelos = vuelos_raw.groupBy("icao_empresa").count()
sin_empresa = emp_vuelos.join(empresas_icao, emp_vuelos.icao_empresa == empresas_icao.icao, "left_anti")
print("Empresas en los vuelos:", emp_vuelos.count(), "| sin datos en ANAC:", sin_empresa.count())
sin_empresa.orderBy(F.desc("count")).show(truncate=False)

aerop_vuelos = (vuelos_raw.select(F.col("icao_origem").alias("icao"))
                .union(vuelos_raw.select(F.col("icao_destino").alias("icao")))
                .groupBy("icao").count())
print("Aeropuertos distintos en los vuelos:", aerop_vuelos.count())
for c in ["gps_code", "ident", "icao_code"]:
    claves = airports_raw.select(F.col(c).alias("icao")).filter(F.col("icao").isNotNull())
    faltan = aerop_vuelos.join(claves, "icao", "left_anti")
    print(f"Cruce por {c}: no cruzan {faltan.count()} aeropuertos, "
          f"{faltan.agg(F.sum('count')).first()[0] or 0} movimientos")
anac = aerodromos_raw.select(F.col("codigo_oaci").alias("icao"))
print("Aeropuertos de los vuelos que están en el cadastro de ANAC:", aerop_vuelos.join(anac, "icao").count())

Empresas en los vuelos: 111 | sin datos en ANAC: 8
+------------+-----+
|icao_empresa|count|
+------------+-----+
|PTB         |2237 |
|PAM         |568  |
|1ED         |57   |
|EUS         |6    |
|LYC         |5    |
|RUC         |4    |
|GJW         |3    |
|CSB         |3    |
+------------+-----+

Aeropuertos distintos en los vuelos: 419
Cruce por gps_code: no cruzan 4 aeropuertos, 28 movimientos
Cruce por ident: no cruzan 12 aeropuertos, 10781 movimientos
Cruce por icao_code: no cruzan 32 aeropuertos, 6097 movimientos
Aeropuertos de los vuelos que están en el cadastro de ANAC: 170


### Hallazgos de la exploración

Los conteos que dicen "en el año" son de los 12 meses; los demás se midieron en enero.

| Hallazgo | Consecuencia |
|---|---|
| Las fechas sin dato vienen con el **texto "null"**, no vacías | Se tratan como nulo en el refinado |
| 28.198 vuelos realizados **no tienen salida prevista** en el año (2.253 en enero); todos tienen código DI distinto de 0 y se tratan como vuelos no programados | No tienen atraso: quedan fuera de la puntualidad, pero cuentan como realizados. La fecha del vuelo es la salida real |
| En el año, 66.695 salidas previstas y 66.695 llegadas previstas (unas 2.700 en enero) tienen **fracción de segundo** ("01:40:00.100000000") | Se toma la fecha hasta los segundos |
| 553 vuelos de pasajeros realizados figuran, en el año, saliendo más de 1 hora antes de lo previsto (24 de ellos, más de 24 horas antes) | Se marcan como atraso extremo (regla 3): si no, contarían como puntuales. La tabla de franjas de la sección 4 justifica el corte en 60 minutos |
| La clave empresa + número + salida prevista + origen **se repetía 825 veces**, por los vuelos sin hora prevista | La clave usa la fecha de referencia y agrega el destino. Con el año completo igual quedan 57 claves repetidas: las resuelve la regla 1 |
| **Código Justificativa** es "N/A" en todos los vuelos | La columna no sirve para explicar cancelaciones |
| Cada archivo mensual trae unos 25 vuelos con fecha de otro mes, casi siempre el siguiente (334 en el año) | Quedan en la partición de su propio mes. Solo un vuelo aparece en dos archivos (ARG 1249, julio y agosto) y lo saca la eliminación de duplicados completos. Los 31 vuelos de enero de 2026, que vienen en el archivo de diciembre, se descartan porque el estudio es 2025 |
| Las listas de empresas de ANAC tienen **solo las activas hoy**: PTB (VoePass) y PAM no están | Sus vuelos se mantienen con nacionalidad DESCONOCIDA |
| Solo 19 de 729 empresas brasileñas tienen código ICAO (709 lo tienen vacío y una tiene un valor basura) | Las demás (taxis aéreos, fumigación) no aparecen en los vuelos |
| Los aeropuertos cruzan mejor con **gps_code** de OurAirports que con ident o icao_code | Se usa gps_code y, si no cruza, ident |
| Teléfono y email de las empresas incluyen mails personales | No pasan a la zona refinada |
| 463 vuelos cancelados tienen tipo_linha **"NA"**, la mayoría del 5 y 6 de abril y del 23 al 26 de mayo | Si se excluyen, las cancelaciones de pasajeros quedan un 12% por debajo en mayo y un 5% en abril (P5). Se toma el tipo del mismo vuelo en otras fechas (regla 4). El desglose exacto, mes a mes, está en la sección 6 |

## 4. Reglas de negocio y refinado

| # | Regla | Dónde se aplica |
|---|---|---|
| 1 | Clave del vuelo: empresa + número + fecha de referencia + origen + destino. Se eliminan las filas duplicadas completas. Si una clave sigue repetida (57 en el año, casi siempre con distinto horario real), queda el registro realizado y, entre esos, el de menor atraso de salida | vuelos |
| 2 | Las fechas vienen con el texto "null" cuando no hay dato. Los cancelados no tienen hora real: su atraso queda nulo y no se imputa. Los vuelos sin hora prevista (todos con código DI distinto de 0) se tratan como no programados: no tienen atraso y quedan fuera de la puntualidad, pero cuentan como realizados. La fecha de referencia del vuelo es la prevista o, si falta, la real. "No informado" queda fuera de las tasas (en 2025 no hay ninguno). Solo quedan los vuelos con fecha de referencia en 2025 | vuelos |
| 3 | Atraso = hora real − hora prevista, en minutos. Negativos (adelantos) se mantienen hasta 60 minutos. Una salida adelantada más de 60 minutos, o un atraso de más de 24 h en valor absoluto, se marca `atraso_extremo` y el atraso pasa a nulo: son errores de registro (hay salidas anotadas 23 horas antes de lo previsto) | vuelos |
| 4 | Solo vuelos de pasajeros N e I en el modelo. En refined quedan todos, con el tipo. En el modelo, los cancelados con tipo "NA" toman el tipo más frecuente del mismo vuelo (empresa, número, origen y destino) en otras fechas; si no aparece en otra fecha, quedan afuera | vuelos → fact |
| 5 | Empresas brasileñas y extranjeras en una sola tabla con `nacionalidad`. Una empresa de los vuelos que no está en ANAC queda como DESCONOCIDA, sin borrar sus vuelos | empresas → dim |
| 6 | Aeropuertos: se cruzan con OurAirports por `gps_code` (y si no, por `ident`). Lo que no cruza queda con tamaño DESCONOCIDO | dim_aeropuerto |
| 7 | Limpieza: trim, valores basura a nulo, tipos correctos, Latin-1 en aeródromos. Teléfono, email y dirección no pasan a refined (datos personales) | todas |
| 8 | Hora local: ANAC informa en hora de Brasília. Brasil no tiene horario de verano desde 2019, así que cada aeropuerto brasileño tiene un desfase fijo según su estado | dim_aeropuerto, fact |

In [12]:
def limpiar(df, columnas, nulos=("",)):
    for c in columnas:
        t = F.trim(F.col(c))
        df = df.withColumn(c, F.when(t.isNull() | t.isin(list(nulos)), None).otherwise(t))
    return df

# Algunas fechas vienen con fracción de segundo ("2025-01-04 01:40:00.100000000"); se toman los
# primeros 19 caracteres (hasta los segundos), que es la precisión del resto del archivo.
ts = lambda c: F.to_timestamp(F.substring(F.col(c), 1, 19), "yyyy-MM-dd HH:mm:ss")
minutos = lambda a, b: (F.unix_timestamp(a) - F.unix_timestamp(b)) / 60.0

vuelos = (limpiar(vuelos_raw, COLS_VRA, nulos=("", "N/A", "null"))
          .withColumn("situacao_voo", F.upper("situacao_voo"))
          .withColumn("partida_prevista", ts("partida_prevista"))
          .withColumn("partida_real", ts("partida_real"))
          .withColumn("chegada_prevista", ts("chegada_prevista"))
          .withColumn("chegada_real", ts("chegada_real"))
          .withColumn("fecha_referencia", F.coalesce("partida_prevista", "partida_real"))
          .filter(F.year("fecha_referencia") == 2025)   # el archivo de diciembre trae 31 vuelos de 2026
          .withColumn("tiene_horario_previsto", F.col("partida_prevista").isNotNull())
          .withColumn("realizado", F.col("situacao_voo") == "REALIZADO")
          .withColumn("cancelado", F.col("situacao_voo") == "CANCELADO")
          .withColumn("informado", F.col("realizado") | F.col("cancelado"))
          .withColumn("atraso_partida_min", F.when(F.col("realizado"), minutos("partida_real", "partida_prevista")))
          .withColumn("atraso_chegada_min", F.when(F.col("realizado"), minutos("chegada_real", "chegada_prevista")))
          .withColumn("atraso_extremo", ~F.col("atraso_partida_min").between(-60, 1440) | (F.abs("atraso_chegada_min") > 1440))
          .withColumn("atraso_extremo", F.coalesce("atraso_extremo", F.lit(False)))
          .withColumn("atraso_partida_min", F.when(~F.col("atraso_extremo"), F.col("atraso_partida_min")))
          .withColumn("atraso_chegada_min", F.when(~F.col("atraso_extremo"), F.col("atraso_chegada_min")))
          .withColumn("anio_mes", F.date_format("fecha_referencia", "yyyy-MM")))

clave_vuelo = ["icao_empresa", "numero_voo", "fecha_referencia", "icao_origem", "icao_destino"]
antes = vuelos.count()
vuelos = vuelos.dropDuplicates([c for c in vuelos.columns if c != "archivo_origen"])
print(f"Filas: {antes:,} | duplicados completos eliminados: {antes - vuelos.count():,}")
repetidas = vuelos.groupBy(*clave_vuelo).count().filter("count > 1")
print("Claves de vuelo repetidas:", repetidas.count())
repetidas.show(5, truncate=False)
# Si la clave sigue repetida, queda el realizado y, entre esos, el de menor atraso de salida (regla 1)
w = (Window.partitionBy(*clave_vuelo)
     .orderBy(F.desc("realizado"), F.abs("atraso_partida_min").asc_nulls_last(), "partida_real"))
vuelos = vuelos.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn").cache()
print("Claves repetidas tras elegir un registro por clave:",
      vuelos.groupBy(*clave_vuelo).count().filter("count > 1").count())
print("Vuelos sin fecha de referencia:", vuelos.filter(F.col("fecha_referencia").isNull()).count())
print("Vuelos sin horario previsto (no programados):", vuelos.filter(~F.col("tiene_horario_previsto")).count())
print("Vuelos con atraso extremo (salida adelantada más de 1 h, o más de 24 h):", vuelos.filter("atraso_extremo").count())
vuelos.groupBy("situacao_voo", "informado").count().show()
vuelos.select(F.min("atraso_partida_min"), F.expr("percentile_approx(atraso_partida_min, 0.5)"),
              F.avg("atraso_partida_min"), F.max("atraso_partida_min")).show()

Filas: 1,005,776 | duplicados completos eliminados: 4
Claves de vuelo repetidas: 57
+------------+----------+-------------------+-----------+------------+-----+
|icao_empresa|numero_voo|fecha_referencia   |icao_origem|icao_destino|count|
+------------+----------+-------------------+-----------+------------+-----+
|ACA         |0090      |2025-11-26 00:35:00|CYYZ       |SBGR        |2    |
|AVA         |Z0261     |2025-10-25 23:50:00|SKBO       |SBGL        |2    |
|ACA         |0090      |2025-10-29 00:35:00|CYYZ       |SBGR        |2    |
|GLO         |9603      |2025-11-08 00:05:00|SBSL       |SBGR        |2    |
|ACA         |Z0090     |2025-10-27 23:00:00|CYYZ       |SBGR        |2    |
+------------+----------+-------------------+-----------+------------+-----+
only showing top 5 rows

Claves repetidas tras elegir un registro por clave: 0
Vuelos sin fecha de referencia: 0
Vuelos sin horario previsto (no programados): 28188
Vuelos con atraso extremo (salida adelantada más de 1 h, o

### Por qué el corte de las salidas adelantadas está en 60 minutos

La regla 3 deja las salidas adelantadas hasta 60 minutos, con atraso negativo, y manda el resto a atraso extremo. La tabla reparte los vuelos de pasajeros realizados que salieron antes de lo previsto según cuánto se adelantaron. El adelanto se recalcula acá desde los horarios, porque en `vuelos` el atraso de los extremos ya quedó nulo.

Si casi todos los adelantos son de pocos minutos y después aparece un grupo que figura horas antes, ese grupo son errores de registro y no vuelos puntuales.

In [13]:
# Salidas adelantadas de vuelos de pasajeros realizados, por franja (regla 3)
adelantos = (vuelos
    .filter(F.col("realizado") & F.col("tiene_horario_previsto") & F.col("tipo_linha").isin("N", "I"))
    .withColumn("adelanto_min", -minutos("partida_real", "partida_prevista"))
    .filter("adelanto_min > 0")
    .withColumn("franja", F.when(F.col("adelanto_min") <= 30, "1. hasta 30 min")
                           .when(F.col("adelanto_min") <= 60, "2. de 30 a 60 min")
                           .when(F.col("adelanto_min") <= 180, "3. de 1 a 3 h")
                           .otherwise("4. más de 3 h"))
    .withColumn("tratamiento", F.when(F.col("adelanto_min") <= 60, "cuenta, con atraso negativo")
                                .otherwise("atraso extremo")))
total_adelantos = adelantos.count()
(adelantos.groupBy("franja", "tratamiento").count()
 .withColumn("pct", F.round(100 * F.col("count") / total_adelantos, 2))
 .orderBy("franja").show(truncate=False))
print(f"Salidas adelantadas: {total_adelantos:,} | más de 24 h antes: {adelantos.filter('adelanto_min > 1440').count():,}")

+-----------------+---------------------------+------+-----+
|franja           |tratamiento                |count |pct  |
+-----------------+---------------------------+------+-----+
|1. hasta 30 min  |cuenta, con atraso negativo|504155|99.61|
|2. de 30 a 60 min|cuenta, con atraso negativo|1415  |0.28 |
|3. de 1 a 3 h    |atraso extremo             |327   |0.06 |
|4. más de 3 h    |atraso extremo             |226   |0.04 |
+-----------------+---------------------------+------+-----+

Salidas adelantadas: 506,123 | más de 24 h antes: 24


Validación a mano del atraso: el vuelo AEA 0057 tiene que dar 50 minutos de atraso de salida.

In [14]:
vuelos.filter((F.col("icao_empresa") == "AEA") & (F.col("numero_voo") == "0057")
              & (F.col("partida_prevista") == F.lit("2025-01-29 19:55:00").cast("timestamp"))) \
      .select("icao_empresa", "numero_voo", "partida_prevista", "partida_real", "atraso_partida_min").show()

+------------+----------+-------------------+-------------------+------------------+
|icao_empresa|numero_voo|   partida_prevista|       partida_real|atraso_partida_min|
+------------+----------+-------------------+-------------------+------------------+
|         AEA|      0057|2025-01-29 19:55:00|2025-01-29 20:45:00|              50.0|
+------------+----------+-------------------+-------------------+------------------+



In [15]:
# Empresas: una sola tabla, sin datos personales
NULOS_EMP = ("", "N/I", "..", ".", "-", "null", "***", "**", "0", "00", "000", "0000000", "00.000-000")
def refinar_empresas(df, nacionalidad):
    return (limpiar(df, COLS_EMPRESAS, NULOS_EMP)
            .withColumn("nacionalidad", F.lit(nacionalidad))
            .withColumn("data_instrumento",
                        F.to_date(F.to_timestamp("data_instrumento", "M/d/yyyy h:mm:ss a")))
            .select(F.col("icao").alias("icao_empresa"), F.col("iata").alias("iata_empresa"),
                    "razao_social", "servico", "nacionalidad", "instrumento", "data_instrumento", "ativa"))

empresas = refinar_empresas(emp_br_raw, "BRASILEÑA").unionByName(refinar_empresas(emp_ext_raw, "EXTRANJERA"))
empresas.groupBy("nacionalidad", F.col("icao_empresa").isNotNull().alias("tiene_icao")).count().show()

+------------+----------+-----+
|nacionalidad|tiene_icao|count|
+------------+----------+-----+
|   BRASILEÑA|      true|   19|
|   BRASILEÑA|     false|  710|
|  EXTRANJERA|     false|    1|
|  EXTRANJERA|      true|  148|
+------------+----------+-----+



In [16]:
# Aeródromos de ANAC: estado como sigla, coordenadas y altitud numéricas
UF_SIGLA = {"ACRE": "AC", "ALAGOAS": "AL", "AMAPA": "AP", "AMAZONAS": "AM", "BAHIA": "BA", "CEARA": "CE",
            "DISTRITO FEDERAL": "DF", "ESPIRITO SANTO": "ES", "GOIAS": "GO", "MARANHAO": "MA",
            "MATO GROSSO": "MT", "MATO GROSSO DO SUL": "MS", "MINAS GERAIS": "MG", "PARA": "PA",
            "PARAIBA": "PB", "PARANA": "PR", "PERNAMBUCO": "PE", "PIAUI": "PI", "RIO DE JANEIRO": "RJ",
            "RIO GRANDE DO NORTE": "RN", "RIO GRANDE DO SUL": "RS", "RONDONIA": "RO", "RORAIMA": "RR",
            "SANTA CATARINA": "SC", "SAO PAULO": "SP", "SERGIPE": "SE", "TOCANTINS": "TO"}
mapa_uf = F.create_map(*[F.lit(x) for kv in UF_SIGLA.items() for x in kv])
sin_tildes = lambda c: F.upper(F.translate(F.col(c), "ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç", "AAAAEEIOOOUCaaaaeeiooouc"))

aerodromos = (limpiar(aerodromos_raw, COLS_AERODROMOS)
              .withColumn("uf_sigla", F.coalesce(mapa_uf[sin_tildes("uf")], mapa_uf[sin_tildes("uf_servido")]))
              .withColumn("latitud", F.col("lat_geopoint").cast("double"))
              .withColumn("longitud", F.col("lon_geopoint").cast("double"))
              .withColumn("altitud_m", F.regexp_replace("altitude", ",", ".").cast("double"))
              .select("codigo_oaci", "ciad", "nome", "municipio", "uf", "uf_sigla", "municipio_servido",
                      "uf_servido", "latitud", "longitud", "altitud_m", "operacao_diurna",
                      "operacao_noturna", "situacao"))
print("Aeródromos sin sigla de estado:", aerodromos.filter("uf_sigla IS NULL").count())
aerodromos.filter("uf_sigla IS NULL").select("codigo_oaci", "nome", "uf", "uf_servido").show(truncate=False)

Aeródromos sin sigla de estado: 0
+-----------+----+---+----------+
|codigo_oaci|nome|uf |uf_servido|
+-----------+----+---+----------+
+-----------+----+---+----------+



In [17]:
# OurAirports: solo las columnas útiles, con tipos
airports = (airports_raw
            .select(F.col("id").cast("long").alias("id_ourairports"), "ident", "type", "name",
                    F.col("latitude_deg").cast("double").alias("latitud"),
                    F.col("longitude_deg").cast("double").alias("longitud"),
                    F.col("elevation_ft").cast("int").alias("elevacion_ft"),
                    "continent", "iso_country", "iso_region", "municipality", "scheduled_service",
                    "gps_code", "icao_code", "iata_code"))
countries = countries_raw.select(F.col("code").alias("iso_pais"), F.col("name").alias("nombre_pais"), "continent")

## 5. Guardado en la zona refinada y control de filas

Todo en parquet bajo `/rfn/vuelos_brasil/v1/`. Después se vuelve a leer cada tabla y se compara la cantidad de filas con la del DataFrame, como en el Práctico 4.

In [18]:
refinadas = {"vuelos": vuelos, "empresas": empresas, "aerodromos": aerodromos,
             "aeropuertos_mundo": airports, "paises": countries}

for nombre, df in refinadas.items():
    df.write.mode("overwrite").parquet(f"{RFN}/{nombre}")

for nombre, df in refinadas.items():
    escrito, leido = df.count(), spark.read.parquet(f"{RFN}/{nombre}").count()
    print(f"[{'OK' if escrito == leido else 'DIFERENTE'}] {nombre}: {escrito:,} filas escritas, {leido:,} leídas")

[OK] vuelos: 1,005,715 filas escritas, 1,005,715 leídas
[OK] empresas: 878 filas escritas, 878 leídas
[OK] aerodromos: 496 filas escritas, 496 leídas
[OK] aeropuertos_mundo: 86,351 filas escritas, 86,351 leídas
[OK] paises: 249 filas escritas, 249 leídas


## 6. Modelo estrella

Las cinco preguntas son agregaciones de vuelos cortadas por empresa, aeropuerto, país y tiempo, así que se eligió un **modelo estrella**: una tabla de hechos con un vuelo por fila y cuatro dimensiones. Resuelve cada pregunta con pocos joins. El modelo normalizado obligaría a más joins para lo mismo; Data Vault agrega complejidad que no hace falta para una carga anual; y OBT repetiría los datos de empresa y aeropuerto en cada vuelo.

| Tabla | Grano | Contenido |
|---|---|---|
| `fact_vuelo` | un vuelo de pasajeros (N o I), programado o no | claves, horarios, atrasos, indicadores; particionada por `anio_mes` |
| `dim_empresa` | una aerolínea | ICAO, IATA, nombre, nacionalidad |
| `dim_aeropuerto` | un aeropuerto (se usa como origen y como destino) | nombre, tamaño, ciudad, estado, país, huso horario |
| `dim_pais` | un país | ISO, nombre, continente |
| `dim_fecha` | un día | año, mes, trimestre, día de la semana, semana |

Qué usa cada pregunta:

| Pregunta | Tablas | Columnas clave |
|---|---|---|
| P1 puntualidad por aerolínea, T1 contra T4, nacional e internacional | fact, dim_empresa, dim_fecha | puntual, realizado, tipo_linha, trimestre |
| P2 rutas con más cancelaciones y su concentración | fact, dim_empresa | icao_origem, icao_destino, cancelado, informado |
| P3 atraso por hora y día en los 10 aeropuertos con más movimiento | fact, dim_aeropuerto | hora_local, dia_semana_local, atraso_partida_min |
| P4 frecuencias por país, T1 contra T4, brasileñas contra extranjeras | fact, dim_aeropuerto, dim_pais, dim_empresa, dim_fecha | iso_pais, nacionalidad, trimestre, realizado |
| P5 puntualidad y cancelación por tamaño de aeropuerto y mes | fact, dim_aeropuerto | tamano, anio_mes, puntual, cancelado |

In [19]:
# dim_empresa: empresas de ANAC con ICAO + las de los vuelos que no están (DESCONOCIDA)
w = Window.partitionBy("icao_empresa").orderBy(F.when(F.col("nacionalidad") == "BRASILEÑA", 0).otherwise(1))
dim_empresa = (empresas.filter(F.col("icao_empresa").isNotNull())
               .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
               .select("icao_empresa", "iata_empresa", F.col("razao_social").alias("nombre_empresa"),
                       "nacionalidad", "servico"))
faltantes = (vuelos.select("icao_empresa").distinct()
             .join(dim_empresa.select("icao_empresa"), "icao_empresa", "left_anti")
             .select("icao_empresa", F.lit(None).cast("string").alias("iata_empresa"),
                     F.lit(None).cast("string").alias("nombre_empresa"),
                     F.lit("DESCONOCIDA").alias("nacionalidad"), F.lit(None).cast("string").alias("servico")))
dim_empresa = dim_empresa.unionByName(faltantes)
dim_empresa.groupBy("nacionalidad").count().show()
faltantes.show()

+------------+-----+
|nacionalidad|count|
+------------+-----+
|  EXTRANJERA|  148|
|   BRASILEÑA|   19|
| DESCONOCIDA|    8|
+------------+-----+

+------------+------------+--------------+------------+-------+
|icao_empresa|iata_empresa|nombre_empresa|nacionalidad|servico|
+------------+------------+--------------+------------+-------+
|         RUC|        NULL|          NULL| DESCONOCIDA|   NULL|
|         LYC|        NULL|          NULL| DESCONOCIDA|   NULL|
|         EUS|        NULL|          NULL| DESCONOCIDA|   NULL|
|         GJW|        NULL|          NULL| DESCONOCIDA|   NULL|
|         PTB|        NULL|          NULL| DESCONOCIDA|   NULL|
|         1ED|        NULL|          NULL| DESCONOCIDA|   NULL|
|         PAM|        NULL|          NULL| DESCONOCIDA|   NULL|
|         CSB|        NULL|          NULL| DESCONOCIDA|   NULL|
+------------+------------+--------------+------------+-------+



In [20]:
# dim_aeropuerto: códigos de los vuelos + OurAirports (gps_code, si no ident) + cadastro de ANAC
codigos = (vuelos.select(F.col("icao_origem").alias("icao_aeropuerto"))
           .union(vuelos.select(F.col("icao_destino").alias("icao_aeropuerto"))).distinct())

claves_oa = (airports.filter(F.col("gps_code").isNotNull()).withColumn("clave", F.col("gps_code")).withColumn("prio", F.lit(1))
             .unionByName(airports.withColumn("clave", F.col("ident")).withColumn("prio", F.lit(2))))
# Si dos aeropuertos comparten gps_code, gana el que además lo tiene como ident y después el de menor id:
# así cada corrida elige el mismo y el tamaño del aeropuerto (P5) no cambia de una corrida a otra
w = Window.partitionBy("clave").orderBy("prio", (F.col("ident") == F.col("clave")).desc(), "id_ourairports")
claves_oa = claves_oa.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn", "prio")

TAMANO = {"large_airport": "GRANDE", "medium_airport": "MEDIANO", "small_airport": "CHICO"}
mapa_tamano = F.create_map(*[F.lit(x) for kv in TAMANO.items() for x in kv])
# Husos fijos de 2025. Por defecto UTC-3 (hora de Brasília).
HUSO_UF = {"AM": -4, "MT": -4, "MS": -4, "RO": -4, "RR": -4, "AC": -5}
HUSO_EXCEPCION = {"SBFN": -2,   # Fernando de Noronha (figura en PE)
                  "SBTT": -5,   # Tabatinga (oeste de AM) - verificar
                  "SWEI": -5}   # Eirunepé (oeste de AM) - verificar
mapa_huso_uf = F.create_map(*[F.lit(x) for kv in HUSO_UF.items() for x in kv])
mapa_huso_exc = F.create_map(*[F.lit(x) for kv in HUSO_EXCEPCION.items() for x in kv])

a = aerodromos.select(F.col("codigo_oaci").alias("icao_aeropuerto"), F.col("nome").alias("nombre_anac"),
                      F.col("municipio").alias("municipio_anac"), "uf_sigla")
dim_aeropuerto = (codigos
    .join(claves_oa, codigos.icao_aeropuerto == claves_oa.clave, "left")
    .join(a, "icao_aeropuerto", "left")
    .withColumn("iso_pais", F.when(F.col("iso_country").isNotNull(), F.col("iso_country"))
                             .when(F.col("uf_sigla").isNotNull(), F.lit("BR")))
    .withColumn("uf", F.coalesce(F.when(F.col("iso_country") == "BR", F.substring("iso_region", 4, 2)), F.col("uf_sigla")))
    .withColumn("huso_utc", F.when(F.col("iso_pais") == "BR",
                                   F.coalesce(mapa_huso_exc[F.col("icao_aeropuerto")], mapa_huso_uf[F.col("uf")], F.lit(-3))))
    .withColumn("tamano", F.coalesce(mapa_tamano[F.col("type")],
                                     F.when(F.col("type").isNotNull(), F.lit("OTRO")), F.lit("DESCONOCIDO")))
    .select("icao_aeropuerto", F.coalesce("name", "nombre_anac").alias("nombre_aeropuerto"),
            F.col("type").alias("tipo_ourairports"), "tamano",
            F.coalesce("municipality", "municipio_anac").alias("ciudad"), "uf", "iso_pais",
            "huso_utc", "latitud", "longitud", "iata_code"))
print("Aeropuertos:", dim_aeropuerto.count())
dim_aeropuerto.groupBy("tamano").count().orderBy(F.desc("count")).show()
dim_aeropuerto.filter("iso_pais IS NULL OR tamano = 'DESCONOCIDO'").show(truncate=False)
dim_aeropuerto.filter("iso_pais = 'BR'").groupBy("huso_utc").count().orderBy("huso_utc").show()

Aeropuertos: 419
+-----------+-----+
|     tamano|count|
+-----------+-----+
|     GRANDE|  211|
|    MEDIANO|  127|
|      CHICO|   78|
|DESCONOCIDO|    3|
+-----------+-----+

+---------------+-----------------+----------------+-----------+------+----+--------+--------+-------+--------+---------+
|icao_aeropuerto|nombre_aeropuerto|tipo_ourairports|tamano     |ciudad|uf  |iso_pais|huso_utc|latitud|longitud|iata_code|
+---------------+-----------------+----------------+-----------+------+----+--------+--------+-------+--------+---------+
|SBFE           |NULL             |NULL            |DESCONOCIDO|NULL  |NULL|NULL    |NULL    |NULL   |NULL    |NULL     |
|GVPR           |NULL             |NULL            |DESCONOCIDO|NULL  |NULL|NULL    |NULL    |NULL   |NULL    |NULL     |
|SEQU           |NULL             |NULL            |DESCONOCIDO|NULL  |NULL|NULL    |NULL    |NULL   |NULL    |NULL     |
+---------------+-----------------+----------------+-----------+------+----+--------+-----

In [21]:
# dim_pais y dim_fecha
CONTINENTE = {"AF": "África", "AN": "Antártida", "AS": "Asia", "EU": "Europa",
              "NA": "América del Norte", "OC": "Oceanía", "SA": "América del Sur"}
mapa_cont = F.create_map(*[F.lit(x) for kv in CONTINENTE.items() for x in kv])
dim_pais = countries.select("iso_pais", "nombre_pais", mapa_cont[F.col("continent")].alias("continente"))

DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
mapa_dias = F.create_map(*[F.lit(x) for i, d in enumerate(DIAS, 1) for x in (i, d)])
dia_iso = lambda c: ((F.dayofweek(c) + 5) % 7) + 1          # 1 = lunes ... 7 = domingo
dim_fecha = (vuelos.select(F.to_date("fecha_referencia").alias("fecha")).filter("fecha IS NOT NULL").distinct()
             .select("fecha", F.year("fecha").alias("anio"), F.month("fecha").alias("mes"),
                     F.date_format("fecha", "yyyy-MM").alias("anio_mes"),
                     F.quarter("fecha").alias("trimestre"),
                     dia_iso("fecha").alias("dia_semana_num"),
                     mapa_dias[dia_iso("fecha")].alias("dia_semana"),
                     F.weekofyear("fecha").alias("semana_anio"),
                     (dia_iso("fecha") >= 6).alias("es_fin_de_semana")))
print("Países:", dim_pais.count(), "| Días:", dim_fecha.count())

Países: 249 | Días: 365


### Vuelos con tipo de línea "NA" (regla 4)

Antes de armar `fact_vuelo` se completa el tipo de los vuelos que vienen con "NA", casi todos cancelados: toman el tipo más frecuente del mismo vuelo (empresa, número, origen y destino) en otras fechas. La tabla muestra cuántos hay, por mes, y qué pasa con cada uno: entra como N o como I, o queda afuera porque el mismo vuelo en otras fechas es de carga (C o G), o porque no aparece en ninguna otra fecha y no hay de dónde sacar el tipo.

In [22]:
# Vuelos con tipo_linha "NA": toman el tipo más frecuente del mismo vuelo en otras fechas (regla 4)
ruta_vuelo = ["icao_empresa", "numero_voo", "icao_origem", "icao_destino"]
w_tipo = Window.partitionBy(*ruta_vuelo).orderBy(F.desc("count"), "tipo_linha")
tipo_conocido = (vuelos.filter(F.col("tipo_linha") != "NA").groupBy(*ruta_vuelo, "tipo_linha").count()
                 .withColumn("rn", F.row_number().over(w_tipo)).filter("rn = 1")
                 .select(*ruta_vuelo, F.col("tipo_linha").alias("tipo_inferido")))

desglose_na = (vuelos.filter(F.col("tipo_linha") == "NA").join(tipo_conocido, ruta_vuelo, "left")
    .withColumn("resultado", F.when(F.col("tipo_inferido") == "N", "entra como N")
                              .when(F.col("tipo_inferido") == "I", "entra como I")
                              .when(F.col("tipo_inferido").isin("C", "G"), "afuera: es de carga")
                              .when(F.col("tipo_inferido").isNull(), "afuera: sin otra fecha")
                              .otherwise(F.concat(F.lit("afuera: tipo "), F.col("tipo_inferido")))))
print("Vuelos con tipo_linha NA:", desglose_na.count())
desglose_na.groupBy("situacao_voo", "resultado").count().orderBy("situacao_voo", "resultado").show(truncate=False)
(desglose_na.groupBy("anio_mes").pivot("resultado").count().na.fill(0)
 .orderBy("anio_mes").show(truncate=False))

Vuelos con tipo_linha NA: 463
+------------+----------------------+-----+
|situacao_voo|resultado             |count|
+------------+----------------------+-----+
|CANCELADO   |afuera: es de carga   |142  |
|CANCELADO   |afuera: sin otra fecha|17   |
|CANCELADO   |entra como I          |120  |
|CANCELADO   |entra como N          |184  |
+------------+----------------------+-----+

+--------+-------------------+----------------------+------------+------------+
|anio_mes|afuera: es de carga|afuera: sin otra fecha|entra como I|entra como N|
+--------+-------------------+----------------------+------------+------------+
|2025-04 |27                 |4                     |35          |58          |
|2025-05 |104                |13                    |67          |85          |
|2025-06 |9                  |0                     |15          |23          |
|2025-07 |2                  |0                     |3           |18          |
+--------+-------------------+----------------------+----

In [23]:
# fact_vuelo: solo pasajeros (N e I), con hora local de salida
huso_origen = dim_aeropuerto.select(F.col("icao_aeropuerto").alias("icao_origem"), F.col("huso_utc").alias("huso_origen"))
vuelos_tipo = vuelos.join(tipo_conocido, ruta_vuelo, "left").withColumn(
    "tipo_linha", F.when(F.col("tipo_linha") == "NA", F.col("tipo_inferido")).otherwise(F.col("tipo_linha")))
fact_vuelo = (vuelos_tipo.filter(F.col("tipo_linha").isin("N", "I"))
    .join(huso_origen, "icao_origem", "left")
    .withColumn("partida_prevista_local",
                F.when(F.col("huso_origen").isNotNull(),
                       F.col("partida_prevista") + F.expr("make_interval(0,0,0,0, huso_origen + 3, 0, 0)")))
    .withColumn("id_vuelo", F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in clave_vuelo]), 256))
    .withColumn("puntual", F.when(F.col("realizado") & F.col("atraso_partida_min").isNotNull(),
                                  F.col("atraso_partida_min") <= 15))
    .withColumn("atrasado_15", F.when(F.col("puntual").isNotNull(), ~F.col("puntual")))
    .select("id_vuelo", "icao_empresa", "numero_voo", "codigo_di", "tipo_linha",
            (F.col("tipo_linha") == "I").alias("es_internacional"),
            "icao_origem", "icao_destino",
            F.to_date("fecha_referencia").alias("fecha"), "tiene_horario_previsto",
            "partida_prevista", "partida_real", "chegada_prevista", "chegada_real",
            F.hour("partida_prevista").alias("hora_prevista_brasilia"),
            "partida_prevista_local",
            F.hour("partida_prevista_local").alias("hora_local"),
            dia_iso("partida_prevista_local").alias("dia_semana_local"),
            "situacao_voo", "realizado", "cancelado", "informado",
            F.round("atraso_partida_min", 1).alias("atraso_partida_min"),
            F.round("atraso_chegada_min", 1).alias("atraso_chegada_min"),
            "puntual", "atrasado_15", "atraso_extremo",
            "anio_mes")                       # la columna de partición va última
    .cache())                                 # se reusa en los controles y al guardar
print("Vuelos en fact_vuelo:", fact_vuelo.count())
fact_vuelo.groupBy("anio_mes", "tipo_linha").count().orderBy("anio_mes", "tipo_linha").show(30)

Vuelos en fact_vuelo: 968900
+--------+----------+-----+
|anio_mes|tipo_linha|count|
+--------+----------+-----+
| 2025-01|         I|15258|
| 2025-01|         N|70981|
| 2025-02|         I|13559|
| 2025-02|         N|62359|
| 2025-03|         I|13776|
| 2025-03|         N|65278|
| 2025-04|         I|12516|
| 2025-04|         N|65141|
| 2025-05|         I|12387|
| 2025-05|         N|66933|
| 2025-06|         I|12469|
| 2025-06|         N|65498|
| 2025-07|         I|14967|
| 2025-07|         N|69651|
| 2025-08|         I|14126|
| 2025-08|         N|67514|
| 2025-09|         I|13115|
| 2025-09|         N|66208|
| 2025-10|         I|13311|
| 2025-10|         N|69263|
| 2025-11|         I|13388|
| 2025-11|         N|65748|
| 2025-12|         I|14988|
| 2025-12|         N|70466|
+--------+----------+-----+



Comprobación para P3: si los 10 aeropuertos con más movimiento están todos en UTC−3, la hora de Brasília ya es la hora local y la decisión del grupo queda justificada con los datos.

Acá se cuentan todos los vuelos del modelo, cancelados incluidos; el notebook 2 cuenta solo los realizados. Por eso los totales son un poco más altos y Confins y Brasília intercambian el cuarto y el quinto puesto, pero los 10 aeropuertos son los mismos.

In [24]:
movimientos = (fact_vuelo.select(F.col("icao_origem").alias("icao_aeropuerto"))
               .union(fact_vuelo.select(F.col("icao_destino").alias("icao_aeropuerto")))
               .groupBy("icao_aeropuerto").count())
(movimientos.join(dim_aeropuerto, "icao_aeropuerto")
 .select("icao_aeropuerto", "nombre_aeropuerto", "uf", "huso_utc", "tamano", "count")
 .orderBy(F.desc("count")).show(10, truncate=40))

+---------------+----------------------------------------+---+--------+------+------+
|icao_aeropuerto|                       nombre_aeropuerto| uf|huso_utc|tamano| count|
+---------------+----------------------------------------+---+--------+------+------+
|           SBGR|São Paulo/Guarulhos–Governor André Fr...| SP|      -3|GRANDE|280725|
|           SBSP|Congonhas–Deputado Freitas Nobre Airport| SP|      -3|GRANDE|190429|
|           SBGL|Rio de Janeiro Galeão – Tom Jobim Int...| RJ|      -3|GRANDE|118608|
|           SBCF|    Tancredo Neves International Airport| MG|      -3|GRANDE|117459|
|           SBBR|Presidente Juscelino Kubitschek Inter...| DF|      -3|GRANDE|116665|
|           SBKP|         Viracopos International Airport| SP|      -3|GRANDE|116296|
|           SBRF|Recife/Guararapes - Gilberto Freyre I...| PE|      -3|GRANDE| 73804|
|           SBRJ|                   Santos Dumont Airport| RJ|      -3|GRANDE| 58730|
|           SBSV|Deputado Luiz Eduardo Magalhães Inter

### Control del modelo antes de guardarlo

Cada tabla tiene que tener una fila por clave y ninguna clave nula. Si una dimensión tuviera una clave repetida, cada JOIN del notebook 2 multiplicaría los vuelos de esa clave y las tasas saldrían mal sin dar ningún error.

También se controla que cada vuelo encuentre su empresa, sus dos aeropuertos y su fecha en las dimensiones: el notebook 2 usa JOIN, y un vuelo sin pareja desaparecería de las respuestas. Si un control falla, el `assert` frena el notebook antes de guardar.

In [25]:
CLAVES = {"dim_empresa": (dim_empresa, "icao_empresa"), "dim_aeropuerto": (dim_aeropuerto, "icao_aeropuerto"),
          "dim_pais": (dim_pais, "iso_pais"), "dim_fecha": (dim_fecha, "fecha"), "fact_vuelo": (fact_vuelo, "id_vuelo")}
for nombre, (df, clave) in CLAVES.items():
    r = df.agg(F.count(F.lit(1)).alias("filas"), F.countDistinct(clave).alias("claves"),
               F.sum(F.col(clave).isNull().cast("int")).alias("nulas")).first()
    nulas = r["nulas"] or 0
    ok = r["filas"] == r["claves"] and nulas == 0
    print(f"[{'OK' if ok else 'DIFERENTE'}] {nombre}: {r['filas']:,} filas, {r['claves']:,} claves distintas, {nulas} nulas")
    assert ok, f"{nombre}: la clave {clave} no es única o tiene nulos"

# Cada vuelo tiene que encontrar su pareja en las dimensiones que usa el notebook 2
PAREJAS = [("icao_empresa", dim_empresa, "icao_empresa"), ("icao_origem", dim_aeropuerto, "icao_aeropuerto"),
           ("icao_destino", dim_aeropuerto, "icao_aeropuerto"), ("fecha", dim_fecha, "fecha")]
for col_fact, dim, col_dim in PAREJAS:
    sin_pareja = (fact_vuelo.select(F.col(col_fact).alias("k")).distinct()
                  .join(dim.select(F.col(col_dim).alias("k")), "k", "left_anti").count())
    print(f"[{'OK' if sin_pareja == 0 else 'DIFERENTE'}] fact_vuelo.{col_fact}: {sin_pareja} valores sin pareja")
    assert sin_pareja == 0, f"fact_vuelo.{col_fact} tiene valores que no están en la dimensión"

# Informativo: P4 deja afuera los aeropuertos cuyo país no está en dim_pais
print("Aeropuertos con país que no está en dim_pais:",
      dim_aeropuerto.filter(F.col("iso_pais").isNotNull()).join(dim_pais, "iso_pais", "left_anti").count())

[OK] dim_empresa: 175 filas, 175 claves distintas, 0 nulas
[OK] dim_aeropuerto: 419 filas, 419 claves distintas, 0 nulas
[OK] dim_pais: 249 filas, 249 claves distintas, 0 nulas
[OK] dim_fecha: 365 filas, 365 claves distintas, 0 nulas
[OK] fact_vuelo: 968,900 filas, 968,900 claves distintas, 0 nulas
[OK] fact_vuelo.icao_empresa: 0 valores sin pareja
[OK] fact_vuelo.icao_origem: 0 valores sin pareja
[OK] fact_vuelo.icao_destino: 0 valores sin pareja
[OK] fact_vuelo.fecha: 0 valores sin pareja
Aeropuertos con país que no está en dim_pais: 0


In [26]:
# Guardar el modelo en su ubicación dentro de la zona refinada
modelo = {"dim_empresa": dim_empresa, "dim_aeropuerto": dim_aeropuerto, "dim_pais": dim_pais,
          "dim_fecha": dim_fecha}
for nombre, df in modelo.items():
    df.write.mode("overwrite").parquet(f"{MODELO}/{nombre}")
fact_vuelo.write.mode("overwrite").partitionBy("anio_mes").parquet(f"{MODELO}/fact_vuelo")
modelo["fact_vuelo"] = fact_vuelo
print("Modelo guardado en", MODELO)

Modelo guardado en /rfn/vuelos_brasil/v1/modelo


## 7. Tablas de Hive

Todas son **EXTERNAL** con `LOCATION` explícito apuntando al parquet del modelo: si se hace `DROP TABLE`, los datos quedan en HDFS. No se usa CTAS, porque crea tablas internas en `/user/hive/warehouse`.

El DDL se arma a partir del esquema de cada DataFrame, así los tipos de Hive coinciden siempre con los del parquet. `fact_vuelo` se particiona por `anio_mes`; como Spark ya escribió las carpetas `anio_mes=...`, `MSCK REPAIR TABLE` registra las particiones.

Si `CREATE DATABASE` da *Permission denied* sobre `/user`, correr en la VM:
`docker exec namenode hdfs dfs -mkdir -p /user/hive/warehouse` y `docker exec namenode hdfs dfs -chmod -R 777 /user/hive`.

La primera celda muestra por qué las rutas llevan servidor y puerto: el warehouse de Spark apunta a `namenode:9000`, pero HDFS escucha en el `8020`. Con la ruta por defecto, `CREATE DATABASE` fallaba; por eso la base y las tablas se crean con la ruta completa que da `fs.defaultFS`.

In [28]:
print("warehouse de Spark:", spark.conf.get("spark.sql.warehouse.dir"))
print("HDFS de Spark     :", spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS"))

warehouse de Spark: hdfs://namenode:9000/user/hive/warehouse
HDFS de Spark     : hdfs://namenode:8020


In [30]:
def tipo_hive(dt):
    return {"StringType": "STRING", "IntegerType": "INT", "LongType": "BIGINT", "DoubleType": "DOUBLE",
            "BooleanType": "BOOLEAN", "TimestampType": "TIMESTAMP", "DateType": "DATE"}[type(dt).__name__]

def ddl_externa(nombre, df, ubicacion, particion=None):
    cols = ",\n  ".join(f"`{f.name}` {tipo_hive(f.dataType)}" for f in df.schema.fields if f.name != particion)
    part = f"\nPARTITIONED BY (`{particion}` STRING)" if particion else ""
    return (f"CREATE EXTERNAL TABLE {DB}.{nombre} (\n  {cols}\n){part}\n"
            f"STORED AS PARQUET\nLOCATION '{ubicacion}'")

# Rutas completas, con servidor y puerto: el warehouse de Spark apunta a namenode:9000, donde HDFS no escucha
HDFS = spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")   # hdfs://namenode:8020
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB} LOCATION '{HDFS}/user/hive/warehouse/{DB}.db'")
ddls = []
for nombre, df in modelo.items():
    particion = "anio_mes" if nombre == "fact_vuelo" else None
    ddl = ddl_externa(nombre, df, f"{HDFS}{MODELO}/{nombre}", particion)
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{nombre}")   # externa: no borra los datos
    spark.sql(ddl)
    ddls.append(ddl + ";")
spark.sql(f"MSCK REPAIR TABLE {DB}.fact_vuelo")

print("\n\n".join(ddls))   # este texto va al repo como sql/ddl_modelo.sql

CREATE EXTERNAL TABLE vuelos_brasil.dim_empresa (
  `icao_empresa` STRING,
  `iata_empresa` STRING,
  `nombre_empresa` STRING,
  `nacionalidad` STRING,
  `servico` STRING
)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/rfn/vuelos_brasil/v1/modelo/dim_empresa';

CREATE EXTERNAL TABLE vuelos_brasil.dim_aeropuerto (
  `icao_aeropuerto` STRING,
  `nombre_aeropuerto` STRING,
  `tipo_ourairports` STRING,
  `tamano` STRING,
  `ciudad` STRING,
  `uf` STRING,
  `iso_pais` STRING,
  `huso_utc` INT,
  `latitud` DOUBLE,
  `longitud` DOUBLE,
  `iata_code` STRING
)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/rfn/vuelos_brasil/v1/modelo/dim_aeropuerto';

CREATE EXTERNAL TABLE vuelos_brasil.dim_pais (
  `iso_pais` STRING,
  `nombre_pais` STRING,
  `continente` STRING
)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/rfn/vuelos_brasil/v1/modelo/dim_pais';

CREATE EXTERNAL TABLE vuelos_brasil.dim_fecha (
  `fecha` DATE,
  `anio` INT,
  `mes` INT,
  `anio_mes` STRING,
  `trimestre` INT,
  `dia_s

## 8. Validación final en Hive

Para cada tabla: filas en Hive contra filas del DataFrame, tipo de tabla y ubicación. Estas salidas (más una consulta en Hue) son la evidencia para el informe.

In [31]:
spark.sql(f"SHOW TABLES IN {DB}").show()
for nombre, df in modelo.items():
    en_hive, en_df = spark.table(f"{DB}.{nombre}").count(), df.count()
    print(f"[{'OK' if en_hive == en_df else 'DIFERENTE'}] {DB}.{nombre}: Hive {en_hive:,} / DataFrame {en_df:,}")
spark.sql(f"SHOW PARTITIONS {DB}.fact_vuelo").show(24, truncate=False)
(spark.sql(f"DESCRIBE FORMATTED {DB}.fact_vuelo")
 .filter(F.col("col_name").isin("Type", "Location", "Serde Library", "InputFormat"))
 .show(truncate=False))

+-------------+--------------+-----------+
|    namespace|     tableName|isTemporary|
+-------------+--------------+-----------+
|vuelos_brasil|dim_aeropuerto|      false|
|vuelos_brasil|   dim_empresa|      false|
|vuelos_brasil|     dim_fecha|      false|
|vuelos_brasil|      dim_pais|      false|
|vuelos_brasil|    fact_vuelo|      false|
+-------------+--------------+-----------+

[OK] vuelos_brasil.dim_empresa: Hive 175 / DataFrame 175
[OK] vuelos_brasil.dim_aeropuerto: Hive 419 / DataFrame 419
[OK] vuelos_brasil.dim_pais: Hive 249 / DataFrame 249
[OK] vuelos_brasil.dim_fecha: Hive 365 / DataFrame 365
[OK] vuelos_brasil.fact_vuelo: Hive 968,900 / DataFrame 968,900
+----------------+
|partition       |
+----------------+
|anio_mes=2025-01|
|anio_mes=2025-02|
|anio_mes=2025-03|
|anio_mes=2025-04|
|anio_mes=2025-05|
|anio_mes=2025-06|
|anio_mes=2025-07|
|anio_mes=2025-08|
|anio_mes=2025-09|
|anio_mes=2025-10|
|anio_mes=2025-11|
|anio_mes=2025-12|
+----------------+

+-------------+--

Consulta de prueba sobre el modelo, como las que hace el notebook 2. Acá el porcentaje de cancelados se calcula sobre todos los vuelos del modelo, incluidos los no programados; el notebook 2 lo calcula sobre los vuelos programados, y por eso su tasa de cancelación es un poco más alta.

In [3]:
# Consulta de prueba sobre el modelo, como la va a usar el notebook 2
spark.sql(f'''
SELECT e.nacionalidad,
       COUNT(*)                                        AS vuelos,
       ROUND(AVG(CASE WHEN f.cancelado THEN 1 ELSE 0 END) * 100, 2) AS pct_cancelados,
       ROUND(AVG(CASE WHEN f.puntual THEN 1.0 WHEN f.puntual = false THEN 0.0 END) * 100, 2) AS pct_puntuales
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_empresa e ON f.icao_empresa = e.icao_empresa
GROUP BY e.nacionalidad
ORDER BY vuelos DESC
''').show()

+------------+------+--------------+-------------+
|nacionalidad|vuelos|pct_cancelados|pct_puntuales|
+------------+------+--------------+-------------+
|   BRASILEÑA|852409|          1.69|        84.92|
|  EXTRANJERA|113610|          5.57|        72.46|
| DESCONOCIDA|  2881|         11.35|        71.39|
+------------+------+--------------+-------------+

